In [1]:
import torch
import math

In [2]:
import os
os.chdir(r'C:\Users\SANAD\micrograd')
print(os.getcwd())

C:\Users\SANAD\micrograd


In [3]:
words = open('names.txt', 'r').read().splitlines()

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos[0])  # should print '.'
print(len(stoi))  # should print 27

.
27


In [5]:
N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [6]:
p = N[0].float()
p = p / p.sum()
p

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [7]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[ix]

'c'

In [8]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p = p / p.sum()
p

tensor([0.6064, 0.3033, 0.0903])

In [9]:
torch.multinomial(p, num_samples=100, replacement=True, generator=g)

tensor([1, 1, 2, 0, 0, 2, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 2, 0, 0,
        1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1,
        0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 1, 0,
        0, 1, 1, 1])

In [10]:
P = N.float()
print(P.shape)  # should print torch.Size([27, 27])
P = P / P.sum(1, keepdim=True)
print(P.shape)  # should print torch.Size([27, 27])

torch.Size([27, 27])
torch.Size([27, 27])


In [11]:
p.shape

torch.Size([3])

In [12]:
P.sum(1,keepdim = True).shape

torch.Size([27, 1])

In [13]:
P[0].sum()

tensor(1.)

In [53]:
P=(N+1).float()
P = P/P.sum(1,keepdim= True)

In [54]:
g = torch.Generator().manual_seed(2147483647)
for i in range(10):
    ix = 0
    out = []
    while True:
       # p = N[ix].float()
        #p = p / p.sum()
        p=P[ix]
        #p = torch.ones(27)/27.0
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            out.append('.')
            break
        out.append(itos[ix])
    print(''.join(out))

cexzmmzoglkurkicqzktyhwmvmzimjttainrlkfukzkktda.
sfcxvpubjtbhrmgotzx.
iczixqctvujkwptedogkkjemkmmsidguenkbvgynywftbspmhwcivgbvtahlvsu.
dsdxxblnwglhpyiw.
iva.
jwrpfdwipkwzkm.
desu.
firmt.
gbiksjbquabsvath.
kuysxqevicmrbxmcwyhrrjenvxmvifkmwmghfvjzxobomysox.


In [55]:
N = torch.zeros((27, 27), dtype=torch.int32)
n=0 
log_likelihood = 0.0
for w in words[:3]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1
        prob = P[ix1,ix2]
        n+=1
        logprob = torch.log(prob)
        log_likelihood += logprob
       # print(f'{ch1,ch2}:{prob:4f}logprob:{logprob:.4f}')
print(f'{log_likelihood = }')
nll = -log_likelihood
print(f'{nll =}')
print(f'{nll/n}')

log_likelihood = tensor(-40.9701)
nll =tensor(40.9701)
2.5606331825256348


In [56]:
xs


tensor([ 0,  5, 13,  ..., 25, 26, 24])

In [57]:
ys

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [58]:
import torch.nn.functional as F

In [59]:
xenc = F.one_hot(xs,num_classes = 27).float()
xenc

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 1., 0., 0.]])

In [60]:
xenc.shape

torch.Size([228146, 27])

In [61]:
xenc.dtype

torch.float32

In [62]:
W = torch.randn((27,27))
xenc @ W

tensor([[-0.1877, -0.9302,  0.3085,  ...,  0.1822, -0.3725, -1.6329],
        [ 1.8126, -0.0545, -0.1888,  ..., -2.1619,  1.0825, -0.6927],
        [-0.0527, -1.6509, -0.6683,  ...,  0.9552,  0.7833, -0.1850],
        ...,
        [-0.0135, -0.3484, -1.5808,  ...,  0.2760,  0.5932,  1.3911],
        [ 0.2130,  0.6051,  0.2556,  ..., -0.0120,  1.8577, -1.3138],
        [ 0.3503,  0.5846, -1.0597,  ...,  0.3674, -0.2499, -1.0406]])

In [65]:
'''for i in range(100):
    xenc = F.one_hot(xs,num_classes = 27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1,keepdim = True)
    loss = -probs[torch.arange(5),ys].log().mean()
    print(loss.item())
    
    W.grad = None
    loss.backward()

    W+= -50 * W.grad'''

'for i in range(100):\n    xenc = F.one_hot(xs,num_classes = 27).float()\n    logits = xenc @ W\n    counts = logits.exp()\n    probs = counts / counts.sum(1,keepdim = True)\n    loss = -probs[torch.arange(5),ys].log().mean()\n    print(loss.item())\n    \n    W.grad = None\n    loss.backward()\n\n    W+= -50 * W.grad'

In [64]:
probs.shape

torch.Size([228146, 27])

In [45]:
nlls = torch.zeros(5)
for i in range(5):
    x = xs[i].item()
    y = ys[i].item()
    print("-"*30)
    print(f'Bigram example {i+1} : {itos[x]}{itos[y]}(indexes {x},{y}')
    print("Input Neural Nets :", x)
    print("Output Probablities from the Neural Nets :", probs[i])
    print("Label(actual next character): ", probs[i])
    p = probs[i,y]
    print("probablity assigned by the nets to the correct character ; ", p.item())
    logp =torch.log(p)
    print('log Likelihood :',nll.item())
    nll = -logp
    print("Negative Log Likelihood :", nll.item())
    nlls[i] = nll

print("="*30)
print("average negative Log likelihood , i.e loss = ", nll.mean().item())
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27),generator = g,requires_grad = True)    

------------------------------
Bigram example 1 : .e(indexes 0,5
Input Neural Nets : 0
Output Probablities from the Neural Nets : tensor([0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
        0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
        0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459],
       grad_fn=<SelectBackward0>)
Label(actual next character):  tensor([0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
        0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
        0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459],
       grad_fn=<SelectBackward0>)
probablity assigned by the nets to the correct character ;  0.01228625513613224
log Likelihood : 4.201204299926758
Negative Log Likelihood : 4.399273872375488
------------------------------
Bigram example 2 : em(indexes 5,13
Input Neural Nets : 5
Output Probablities from the Neural Nets :

In [46]:
xs

tensor([ 0,  5, 13,  ..., 25, 26, 24])

In [47]:
ys

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [48]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27),generator = g,requires_grad = True)



In [84]:
xenc = F.one_hot(xs,num_classes = 27).float()
logits = xenc @ W
counts = logits.exp()
probs = counts / counts.sum(1,keepdim = True)
num = xs.nelement()
loss = -probs[torch.arange(num),ys].log().mean()
loss


tensor(2.4748, grad_fn=<NegBackward0>)

In [85]:
print(loss.item())

2.474761724472046


In [86]:
xs , ys = [] , []
for w in words[:]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('Number of Examples :',num)
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27),generator = g,requires_grad = True)

Number of Examples : 228146


In [87]:
(W**2).sum()

tensor(704.5897, grad_fn=<SumBackward0>)

In [ ]:
for i in range(100):
    xenc = F.one_hot(xs,num_classes = 27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1,keepdim = True)
    loss = -probs[torch.arange(num),ys].log().mean()+0.01*(W**2).mean()
    print(loss.item())
    
    W.grad = None
    loss.backward()
    learning_rate = 50

    W.data += -learning_rate * W.grad

3.7686190605163574
3.3788068294525146
3.161090850830078
3.027186155319214
2.9344842433929443
2.8672313690185547
2.816654682159424
2.777146577835083
2.745253801345825
2.7188303470611572
2.696505546569824
2.6773719787597656
2.6608052253723145
2.6463515758514404
2.633665084838867
2.622471570968628
2.6125476360321045
2.6037068367004395
2.595794916152954
2.5886809825897217
2.582256317138672
2.5764293670654297
2.5711238384246826
2.566272735595703
2.5618226528167725
2.5577263832092285
2.5539445877075195
2.550442695617676
2.5471925735473633
2.5441696643829346
2.5413525104522705
2.538721799850464
2.536262035369873
2.5339581966400146
2.531797409057617
2.5297679901123047
2.527860164642334
2.5260636806488037
2.5243709087371826
2.522773265838623
2.521263837814331
2.519836664199829
2.5184857845306396
2.517204999923706
2.515990734100342
2.5148372650146484
2.5137410163879395
2.512697696685791
2.511704921722412
2.5107579231262207
2.509854555130005
2.5089924335479736
2.5081686973571777
2.507380485534668

In [82]:
#finally, sample from the 'neural net' model
g = torch.Generator().manual_seed (2147483647)
for i in range(5):
    out = []
    ix = 0
    while True:
#
# BEFORE:
#p= P[ix]
#
# NOW:
       xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
       logits = xenc @ W# predict log-counts
       counts = logits.exp() # counts, equivalent to N
       p = counts / counts.sum(1, keepdims=True) # probabilities for next character
#
       ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
       out.append(itos [ix])
       if ix == 0:
        break
    print(''.join(out))

cexze.
momasurailezityha.
konimittain.
llayn.
ka.
